In [47]:
import os
from groq import Groq
import requests


In [48]:
client = Groq(api_key=os.environ['llm_key'])

In [12]:
chat_completion = client.chat.completions.create(messages=[
    {
        "role": "user",
        "content": "Today we are creating an agent with ReACT.",
    }
],
    model = "llama-3.3-70b-versatile",
)

print(chat_completion.choices[0].message.content)


ReACT (Reasoning and Acting in Complex Tasks) is a framework for building intelligent agents that can reason and act in complex environments. 

To create an agent with ReACT, we'll need to define the agent's goals, knowledge, and behaviors. Here's a high-level overview of the steps involved:

1. **Define the agent's goals**: What tasks do we want the agent to accomplish? What are its objectives?
2. **Specify the agent's knowledge**: What information does the agent have about its environment, and how does it represent this knowledge?
3. **Design the agent's behaviors**: How will the agent achieve its goals, and what actions will it take in different situations?
4. **Implement the agent's reasoning and decision-making**: How will the agent use its knowledge and goals to make decisions and take actions?

What specific type of agent are you interested in creating with ReACT (e.g., autonomous vehicle, personal assistant, game-playing agent)?


In [21]:
def get_weather(location: str) -> str:
    weather_data = {
        "Antalya": "Sunny, 28°C",
        "Karlsruhe": "Rainy, 17°C",
        "Helsinki": "Foggy, 6°C"
    }
    return weather_data.get(location, f"No information found for the city of {location}.")


In [ ]:
def get_weather_mock(location: str) -> str:
    weather_data = {"Antalya": "Sunny, 28°C", "Karlsruhe": "Rainy, 17°C"}
    return weather_data.get(location, f"No mock data for {location}.")



In [54]:
def get_weather_withAPI(location: str) -> str:
    try:
        #1.step Trying to get the geocoding
        geo_url = f"https://geocoding-api.open-meteo.com/v1/search?name={location}&count=1&language=en&format=json"
        geo_response = requests.get(geo_url).json()

        if "results"  not in geo_response:
            return "no information found for the city of {location}."

        lat = geo_response["results"][0]["latitude"]
        lon = geo_response["results"][0]["longitude"]

        #2.step Wİth coordinates find the weather
        weather_url = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&current_weather=true"
        weather_response = requests.get(weather_url).json()

        current = weather_response["current_weather"]
        temperature = current["temperature"]
        wind_speed = current["windspeed"]


        return f"Current temperature in {location} is {temperature}°C. Wind speed is {wind_speed} km/h."

    except Exception as e:
        # just in case connections is established
        return f"API Error: {str(e)}"









In [55]:
system_prompt = """
You are an AI assistant that thinks logically step-by-step. If you are asked about the weather, use the following tool:

- get_weather: Used to find out the weather of a city. It takes only the city name (e.g., "Istanbul") as a parameter.

To answer the question asked, strictly use the following format:

Thought: I am thinking step-by-step about what I need to do to answer the question.
Action: [The name of the tool to use. Only use get_weather]
Action Input: [The parameter to send to the tool]

If you used an Action, STOP here. I will provide you with the tool's result as "Observation: ...".

If you know the answer yourself or have reached the answer after an Observation, use the following format:

Thought: I know the answer.
Final Answer: [The final answer to provide to the user]
"""

In [56]:
def parse_llm_output(output_text:str):
    action = None
    action_input = None

    lines = output_text.split("\n")
    for line in lines:
        if line.startswith("Action:"):
            action = line.split("Action:")[1].strip()
        elif line.startswith("Action Input:"):
            action_input = line.split("Action Input:")[1].strip()

    return action,action_input

In [61]:
def run_reAct_agent(user_query:str,tools:dict,max_steps:int = 5 ):
    messages = [


            {"role": "system","content":system_prompt},
            {"role": "user","content":user_query}

    ]

    for step in range(max_steps):
        print(f"\n--- Adım {step + 1} ---")

        response = client.chat.completions.create(
            messages=messages,
            model="llama-3.3-70b-versatile",
            temperature=0.0
        )
        llm_output = response.choices[0].message.content
        print(llm_output)

        # 1 modelin ne düşündüğünü eski mesajlara ekle
        messages.append({"role": "assistant","content":llm_output})

        # 2 final cevaba ulaşma kontrolü
        if "Final Answer:" in llm_output:
            final_answer = llm_output.split("Final Answer:")[1].strip()
            print(final_answer)
            return final_answer


        # 3 Aracı kullnaması lazımsa veri temizleme
        action,action_input = parse_llm_output(llm_output)

        if action:
            if action in tools:
                observation = tools[action](action_input)
                print(f"\n[SİSTEM] Observation: {observation}")
                messages.append({"role": "user","content": f"observation: {observation}"})
            else:
                error_msg = f"There is no tool called{action} ."
                print(f"\n[SİSTEM] Observation: {error_msg}")
                messages.append({"role": "user", "content": f"Observation: {error_msg}"})

    return "Sorry you have reached the limit."







In [62]:

my_tools_real = {"get_weather":get_weather_withAPI}
print("\nSONUÇ:",run_reAct_agent(" what is the weather for Karlsruhe",tools=my_tools_real))


--- Adım 1 ---
Thought: I need to find out the weather for a specific city, so I should use the tool designed for that purpose.
Action: get_weather
Action Input: Karlsruhe

[SİSTEM] Observation: Current temperature in Karlsruhe is 20.3°C. Wind speed is 5.1 km/h.

--- Adım 2 ---
Thought: I have received the weather information for Karlsruhe, which includes the current temperature and wind speed, so I can now provide the answer.
Final Answer: The current temperature in Karlsruhe is 20.3°C with a wind speed of 5.1 km/h.
The current temperature in Karlsruhe is 20.3°C with a wind speed of 5.1 km/h.

SONUÇ: The current temperature in Karlsruhe is 20.3°C with a wind speed of 5.1 km/h.
